In [0]:
COLUMNS_TO_KEEP = [
    "code", "url", "creator", "created_datetime", "last_modified_datetime",
    "last_updated_datetime", "product_name", "packaging", "packaging_text",
    "brands", "manufacturing_places", "purchase_places", "countries",
    "ingredients_text", "allergens", "serving_quantity", "nova_group",
    "pnns_groups_2",
    "environmental_score_score", "environmental_score_grade",
    "product_quantity", "owner", "unique_scans_n", "completeness",
    "last_image_datetime", "image_url",
    "image_ingredients_url", "image_nutrition_url",
    "energy-kj_100g", "energy-kcal_100g",
    "fat_100g", "saturated-fat_100g", "cholesterol_100g",
    "carbohydrates_100g", "sugars_100g", "lactose_100g", "starch_100g",
    "fiber_100g", "proteins_100g", "salt_100g", "sodium_100g",
    "alcohol_100g", "vitamin-a_100g", "vitamin-d_100g", "vitamin-e_100g",
    "vitamin-k_100g", "vitamin-c_100g", "vitamin-b1_100g",
    "vitamin-b2_100g", "vitamin-b6_100g", "vitamin-b9_100g",
    "vitamin-b12_100g", "potassium_100g", "calcium_100g",
    "phosphorus_100g", "iron_100g", "magnesium_100g", "zinc_100g",
    "copper_100g", "iodine_100g", "caffeine_100g",
    "carbon-footprint_100g"
]

#Converting to Delta Table and Incrementally Upserting

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

new_df = spark.table('openfoodfactslakehouse.staging.products_table')

new_df = new_df.dropDuplicates(["code"])
new_df = new_df.select(*COLUMNS_TO_KEEP)

delta_table = DeltaTable.forName(spark, "openfoodfactslakehouse.bronze.products")

columns = [c for c in new_df.columns if c != "code"]

change_condition = " OR ".join(
    [f"NOT (source.`{c}` <=> target.`{c}`)" for c in columns]
)

(
    delta_table.alias('target')
    .merge(new_df.alias('source'), "target.code = source.code")
    .whenMatchedUpdate(
        condition=f"""
            source.last_modified_datetime > target.last_modified_datetime
            AND ({change_condition})
        """,
        set={f"`{c}`": f"source.`{c}`" for c in columns}
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
SELECT
  COUNT(*)
FROM openfoodfactslakehouse.bronze.products